# Transfer Learning: 사전 학습 모델을 새로운 데이터에 적용하기

이 노트북에서는 ImageNet으로 사전 학습된 ResNet-18을 새로운 이미지 분류 문제에 적용한다. 단순한 코드 사용법뿐 아니라 **왜 전이학습이 효과적인지**, **feature extraction과 fine-tuning이 어떻게 다른지**, **어떤 파라미터가 실제로 업데이트되는지**를 단계별로 확인한다.

학습 목표는 다음과 같다.

1. 사전 학습과 전이학습의 관계를 이해한다.
2. ImageNet용 마지막 출력층을 새로운 클래스 수에 맞게 교체한다.
3. Backbone을 고정한 feature extraction을 구현한다.
4. 전체 또는 일부 backbone을 학습하는 fine-tuning을 구현한다.
5. Validation 기준 best model을 저장하고 새로운 이미지를 추론한다.
6. Domain shift, BatchNorm, 학습률, 과적합과 같은 실무 문제를 이해한다.


## 1. 전이학습이란?

전이학습(transfer learning)은 큰 원본 데이터셋에서 학습한 지식을 새로운 목표 작업에 재사용하는 방법이다.

```text
ImageNet과 같은 대규모 원본 데이터
             ↓ 사전 학습
       범용 이미지 특징
             ↓ 전이
고양이/강아지, 불량품, 교통표지판 등 새로운 작업
```

CNN의 초기 layer는 선, 모서리, 색상 변화처럼 여러 이미지 작업에 공통적인 특징을 학습하는 경향이 있다. 깊은 layer로 갈수록 원본 데이터의 객체 부분과 의미에 특화된 표현을 학습한다. 따라서 새로운 데이터가 많지 않더라도 사전 학습된 특징에서 시작하면 무작위 초기화보다 빠르고 안정적으로 학습할 수 있다.

전이학습이 항상 유리한 것은 아니다. 원본과 목표 데이터의 domain이 매우 다르거나 목표 데이터가 충분히 많으면 처음부터 학습하는 방식도 비교해야 한다.


## 2. 두 가지 대표 전략

### 2.1 고정 Feature Extractor

사전 학습된 convolution backbone을 고정하고 마지막 classifier만 새로 학습한다.

```text
입력 → 사전 학습 Backbone(고정) → 새 Classifier(학습) → 새 클래스 logits
```

- 데이터가 적을 때 유용하다.
- 학습할 파라미터와 메모리 사용량이 적다.
- Backbone 표현이 목표 데이터와 잘 맞지 않으면 성능에 한계가 있다.

### 2.2 Fine-tuning

사전 학습 가중치에서 시작하지만 backbone 일부 또는 전체도 목표 데이터에 맞게 업데이트한다.

```text
입력 → 사전 학습 Backbone(작게 업데이트) → 새 Classifier(크게 업데이트)
```

- 목표 domain에 맞게 특징을 조정할 수 있다.
- Feature extraction보다 더 높은 성능을 낼 가능성이 있다.
- 데이터가 적거나 학습률이 크면 pretrained knowledge가 빠르게 훼손되거나 과적합될 수 있다.

| 조건 | 권장 시작 전략 |
|---|---|
| 데이터가 매우 적고 ImageNet과 유사 | Classifier만 학습 |
| 데이터가 적지만 domain 차이가 큼 | Classifier 학습 후 상위 stage 점진 해제 |
| 데이터가 충분하고 ImageNet과 유사 | 전체 fine-tuning, 작은 backbone LR |
| 데이터가 충분하고 domain 차이가 큼 | 전체 fine-tuning과 scratch baseline 비교 |


## 3. 수학적으로 보는 전이학습

사전 학습 모델을 feature extractor $f$와 classifier $g$로 나누면 다음과 같다.

$$
\mathbf{h}=f(\mathbf{x};\theta_f)
$$

$$
\mathbf{z}=g(\mathbf{h};\theta_g)
$$

- $\theta_f$: backbone 파라미터
- $\theta_g$: classifier 파라미터
- $\mathbf{z}$: 클래스별 logit

Feature extraction에서는 $\theta_f$를 고정하고 $\theta_g$만 최적화한다.

$$
\min_{\theta_g}\mathcal{L}(\theta_f^{pretrained},\theta_g)
$$

전체 fine-tuning에서는 두 파라미터를 모두 최적화한다.

$$
\min_{\theta_f,\theta_g}\mathcal{L}(\theta_f,\theta_g)
$$

일반적으로 새 classifier는 무작위 초기화이고 backbone은 이미 유용한 표현을 갖고 있으므로 서로 다른 학습률을 사용할 수 있다.

$$
\eta_{backbone}<\eta_{classifier}
$$


In [6]:
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.optim import SGD
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import ResNet18_Weights, resnet18


In [7]:
# 재현 가능한 실험을 위한 seed 설정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"device: {device}")


device: mps


## 4. 데이터 폴더 구조

이 노트북은 `torchvision.datasets.ImageFolder`를 사용한다. 폴더 이름이 클래스 이름이 되며 학습·검증 데이터는 다음처럼 분리한다.

```text
transfer_data/
├── train/
│   ├── class_a/
│   │   ├── image001.jpg
│   │   └── image002.jpg
│   └── class_b/
│       └── image003.jpg
└── val/
    ├── class_a/
    └── class_b/
```

`train`과 `val`의 클래스 폴더 이름은 같아야 한다. Test 데이터를 사용한다면 모델과 하이퍼파라미터 선택에 사용하지 말고 최종 평가에만 사용한다.


In [8]:
# 자신의 데이터 위치에 맞게 변경한다.
DATA_DIR = Path("transfer_data")
TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "val"

BATCH_SIZE = 32
NUM_WORKERS = 0  # Notebook/macOS에서는 먼저 0으로 확인한다.
NUM_EPOCHS = 10

print(f"train directory: {TRAIN_DIR.resolve()}")
print(f"validation directory: {VAL_DIR.resolve()}")


train directory: /Users/donghun2/workspace/machine_learning/dacon/06.PretrainedModel/transfer_data/train
validation directory: /Users/donghun2/workspace/machine_learning/dacon/06.PretrainedModel/transfer_data/val


## 5. 사전 학습 Weight에 맞는 전처리

사전 학습 모델은 특정 resize, crop, interpolation, pixel 범위와 정규화를 기준으로 학습된다. 입력 전처리가 다르면 pretrained feature의 의미가 달라져 성능이 감소할 수 있다.

ResNet18 ImageNet weight의 대표적인 RGB 정규화 값은 다음과 같다.

$$
x'_{c}=\frac{x_c-\mu_c}{\sigma_c}
$$

```text
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]
```

Validation에는 무작위 증강을 넣지 않는다. 학습 데이터의 증강도 label 의미를 유지해야 한다. 예를 들어 좌회전·우회전 표지판에는 horizontal flip이 잘못된 label을 만들 수 있다.


In [9]:
weights = ResNet18_Weights.DEFAULT

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    # 좌우 반전이 label을 보존하는 데이터에서만 아래 줄을 활성화한다.
    # transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

# 사전 학습 weight가 제공하는 공식 추론 transform도 확인할 수 있다.
official_inference_transform = weights.transforms()
print(official_inference_transform)


ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [10]:
if not TRAIN_DIR.is_dir() or not VAL_DIR.is_dir():
    raise FileNotFoundError(
        "TRAIN_DIR와 VAL_DIR를 찾을 수 없습니다. "
        "위의 DATA_DIR를 실제 데이터 경로로 수정하세요."
    )

train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform,
)
val_dataset = datasets.ImageFolder(
    VAL_DIR,
    transform=val_transform,
)

if train_dataset.class_to_idx != val_dataset.class_to_idx:
    raise ValueError(
        "train과 val의 클래스 폴더 구성이 다릅니다."
    )

class_names = train_dataset.classes
num_classes = len(class_names)

print(f"classes: {class_names}")
print(f"class_to_idx: {train_dataset.class_to_idx}")
print(f"train samples: {len(train_dataset)}")
print(f"validation samples: {len(val_dataset)}")


FileNotFoundError: TRAIN_DIR와 VAL_DIR를 찾을 수 없습니다. 위의 DATA_DIR를 실제 데이터 경로로 수정하세요.

In [11]:
pin_memory = device.type == "cuda"

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

dataloaders = {
    "train": train_loader,
    "val": val_loader,
}


NameError: name 'train_dataset' is not defined

In [12]:
def denormalize(image_tensor):
    mean = torch.tensor(imagenet_mean).view(3, 1, 1)
    std = torch.tensor(imagenet_std).view(3, 1, 1)
    return image_tensor.cpu() * std + mean


images, labels = next(iter(train_loader))
show_count = min(8, images.size(0))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for axis in axes.flat:
    axis.axis("off")

for index in range(show_count):
    image = denormalize(images[index]).clamp(0, 1)
    image = image.permute(1, 2, 0).numpy()
    axes.flat[index].imshow(image)
    axes.flat[index].set_title(class_names[labels[index].item()])

plt.tight_layout()
plt.show()


NameError: name 'train_loader' is not defined

## 6. 마지막 출력층을 왜 교체하는가?

ImageNet으로 학습한 ResNet-18의 마지막 `fc`는 1000개 ImageNet 클래스의 logit을 출력한다. 새로운 데이터가 $C$개 클래스라면 마지막 layer를 `Linear(in_features, C)`로 교체해야 한다.

```text
ImageNet 모델: [batch_size, 1000]
새 분류 모델: [batch_size, num_classes]
```

각 출력은 확률이 아니라 정규화되지 않은 클래스 점수인 logit이다. `CrossEntropyLoss`가 내부에서 log-softmax에 해당하는 계산을 수행하므로 모델의 마지막에 softmax를 넣지 않는다. 추론에서 확률이 필요할 때만 `logits.softmax(dim=1)`을 사용한다.


In [13]:
base_model = resnet18(weights=weights)
print(base_model.fc)
print(f"ImageNet output classes: {base_model.fc.out_features}")
print(f"Classifier input features: {base_model.fc.in_features}")


Linear(in_features=512, out_features=1000, bias=True)
ImageNet output classes: 1000
Classifier input features: 512


## 7. 전략 A: 고정 Feature Extractor

`parameter.requires_grad = False`로 설정하면 `loss.backward()`가 해당 파라미터의 gradient를 계산하지 않는다. 이후 새 `fc`를 만들면 새 layer는 기본적으로 `requires_grad=True`이므로 classifier만 학습된다.

주의할 점은 파라미터 고정과 `model.eval()`이 서로 다른 기능이라는 것이다.

- `requires_grad=False`: weight gradient와 optimizer update를 중지한다.
- `model.eval()`: Dropout과 BatchNorm을 평가 방식으로 전환한다.

학습 함수에서 전체 모델에 `model.train()`을 호출하면 고정된 backbone의 BatchNorm running statistics는 여전히 바뀔 수 있다. 이 노트북의 첫 실습은 torchvision 공식 튜토리얼과 같은 기본 패턴을 사용하며, 작은 batch나 큰 domain shift에서는 뒤에서 설명하는 BatchNorm 정책을 별도로 검토한다.


In [14]:
feature_model = resnet18(weights=weights)

# 기존 backbone 전체를 고정한다.
for parameter in feature_model.parameters():
    parameter.requires_grad = False

# 기존 fc의 입력 feature 수를 유지하고 출력만 새 클래스 수로 변경한다.
in_features = feature_model.fc.in_features
feature_model.fc = nn.Linear(in_features, num_classes)
feature_model = feature_model.to(device)

criterion = nn.CrossEntropyLoss()

# 실제로 학습되는 새 classifier 파라미터만 optimizer에 전달한다.
feature_optimizer = SGD(
    feature_model.fc.parameters(),
    lr=0.01,
    momentum=0.9,
    weight_decay=1e-4,
)
feature_scheduler = StepLR(
    feature_optimizer,
    step_size=5,
    gamma=0.1,
)

print(feature_model.fc)


NameError: name 'num_classes' is not defined

In [15]:
def count_parameters(model):
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
    return total, trainable


total_params, trainable_params = count_parameters(feature_model)
print(f"total parameters: {total_params:,}")
print(f"trainable parameters: {trainable_params:,}")
print(f"trainable ratio: {trainable_params / total_params:.2%}")

print("\nTrainable parameter names")
for name, parameter in feature_model.named_parameters():
    if parameter.requires_grad:
        print(name, tuple(parameter.shape))


total parameters: 11,689,512
trainable parameters: 0
trainable ratio: 0.00%

Trainable parameter names


## 8. 학습과 검증 함수

`CrossEntropyLoss`의 기본 출력은 현재 batch의 평균 loss다. 마지막 batch가 더 작을 수 있으므로 epoch loss를 정확히 구하려면 batch 평균에 실제 batch size를 곱해 합계를 누적한 뒤 전체 sample 수로 나눈다.

$$
L_{epoch}=\frac{\sum_i L_i}{N}
$$

Accuracy는 전체 sample 중 예측 클래스와 정답이 같은 비율이다.

$$
Accuracy=\frac{\sum_i \mathbf{1}(\hat{y}_i=y_i)}{N}
$$


In [ ]:
def train_one_epoch(model, data_loader, criterion, optimizer, device):
    model.train()

    loss_sum = 0.0
    correct = 0
    sample_count = 0

    for images, labels in data_loader:
        images = images.to(
            device,
            dtype=torch.float32,
            non_blocking=device.type == "cuda",
        )
        labels = labels.to(
            device,
            dtype=torch.long,
            non_blocking=device.type == "cuda",
        )

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        loss_sum += loss.item() * batch_size
        correct += (logits.argmax(dim=1) == labels).sum().item()
        sample_count += batch_size

    return {
        "loss": loss_sum / sample_count,
        "accuracy": correct / sample_count,
    }


In [ ]:
def evaluate(model, data_loader, criterion, device):
    model.eval()

    loss_sum = 0.0
    correct = 0
    sample_count = 0

    with torch.inference_mode():
        for images, labels in data_loader:
            images = images.to(
                device,
                dtype=torch.float32,
                non_blocking=device.type == "cuda",
            )
            labels = labels.to(
                device,
                dtype=torch.long,
                non_blocking=device.type == "cuda",
            )

            logits = model(images)
            loss = criterion(logits, labels)

            batch_size = labels.size(0)
            loss_sum += loss.item() * batch_size
            correct += (logits.argmax(dim=1) == labels).sum().item()
            sample_count += batch_size

    return {
        "loss": loss_sum / sample_count,
        "accuracy": correct / sample_count,
    }


In [ ]:
def fit(
    model,
    dataloaders,
    criterion,
    optimizer,
    device,
    num_epochs,
    scheduler=None,
):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
    }

    best_accuracy = float("-inf")
    best_state = deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        train_metrics = train_one_epoch(
            model,
            dataloaders["train"],
            criterion,
            optimizer,
            device,
        )
        val_metrics = evaluate(
            model,
            dataloaders["val"],
            criterion,
            device,
        )

        if scheduler is not None:
            scheduler.step()

        history["train_loss"].append(train_metrics["loss"])
        history["train_accuracy"].append(train_metrics["accuracy"])
        history["val_loss"].append(val_metrics["loss"])
        history["val_accuracy"].append(val_metrics["accuracy"])

        if val_metrics["accuracy"] > best_accuracy:
            best_accuracy = val_metrics["accuracy"]
            best_state = deepcopy(model.state_dict())

        current_lr = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch {epoch + 1:03d}/{num_epochs:03d} | "
            f"lr={current_lr:.2e} | "
            f"train loss={train_metrics['loss']:.4f}, "
            f"train acc={train_metrics['accuracy']:.2%} | "
            f"val loss={val_metrics['loss']:.4f}, "
            f"val acc={val_metrics['accuracy']:.2%}"
        )

    model.load_state_dict(best_state)
    print(f"best validation accuracy: {best_accuracy:.2%}")

    return model, history


In [ ]:
# 고정 feature extractor 학습
feature_model, feature_history = fit(
    model=feature_model,
    dataloaders=dataloaders,
    criterion=criterion,
    optimizer=feature_optimizer,
    device=device,
    num_epochs=NUM_EPOCHS,
    scheduler=feature_scheduler,
)


## 9. 전략 B: 전체 Fine-tuning

전체 fine-tuning에서는 backbone 파라미터도 gradient를 계산한다. 새 classifier에는 상대적으로 큰 학습률을, pretrained backbone에는 더 작은 학습률을 주는 discriminative learning rate를 사용할 수 있다.

```text
Backbone: 이미 학습된 유용한 특징 → 작은 learning rate
Classifier: 무작위로 새로 생성됨 → 큰 learning rate
```

Feature extraction 결과에서 그대로 이어서 전체를 풀 수도 있지만, 두 전략을 공정하게 비교하기 위해 아래에서는 같은 pretrained weight에서 새 모델을 만든다.


In [ ]:
finetune_model = resnet18(weights=weights)
in_features = finetune_model.fc.in_features
finetune_model.fc = nn.Linear(in_features, num_classes)
finetune_model = finetune_model.to(device)

# fc가 아닌 모든 파라미터를 backbone 그룹으로 구성한다.
backbone_parameters = [
    parameter
    for name, parameter in finetune_model.named_parameters()
    if not name.startswith("fc.")
]

finetune_optimizer = SGD(
    [
        {
            "params": backbone_parameters,
            "lr": 1e-4,
        },
        {
            "params": finetune_model.fc.parameters(),
            "lr": 1e-3,
        },
    ],
    momentum=0.9,
    weight_decay=1e-4,
)
finetune_scheduler = StepLR(
    finetune_optimizer,
    step_size=5,
    gamma=0.1,
)

total_params, trainable_params = count_parameters(finetune_model)
print(f"trainable parameters: {trainable_params:,} / {total_params:,}")
for index, group in enumerate(finetune_optimizer.param_groups):
    print(f"parameter group {index} lr: {group['lr']:.1e}")


In [ ]:
finetune_model, finetune_history = fit(
    model=finetune_model,
    dataloaders=dataloaders,
    criterion=criterion,
    optimizer=finetune_optimizer,
    device=device,
    num_epochs=NUM_EPOCHS,
    scheduler=finetune_scheduler,
)


## 10. 점진적 Unfreezing

데이터가 적거나 전체 fine-tuning이 불안정하면 다음 순서를 사용할 수 있다.

1. Classifier만 학습한다.
2. 가장 깊은 stage인 `layer4`와 classifier를 학습한다.
3. 필요하면 `layer3`까지 푼다.
4. 충분한 데이터와 안정적인 validation 결과가 있을 때 전체를 푼다.

깊은 stage는 task-specific 특징을 많이 포함하는 경향이 있어 먼저 조정하기 좋다. `requires_grad`를 변경한 후에는 optimizer가 새 학습 대상 파라미터를 포함하도록 다시 생성해야 한다.


In [ ]:
gradual_model = resnet18(weights=weights)
gradual_model.fc = nn.Linear(
    gradual_model.fc.in_features,
    num_classes,
)

# 먼저 전체를 고정한다.
for parameter in gradual_model.parameters():
    parameter.requires_grad = False

# 가장 깊은 residual stage와 classifier만 해제한다.
for parameter in gradual_model.layer4.parameters():
    parameter.requires_grad = True
for parameter in gradual_model.fc.parameters():
    parameter.requires_grad = True

gradual_model = gradual_model.to(device)

gradual_optimizer = SGD(
    [
        {"params": gradual_model.layer4.parameters(), "lr": 1e-4},
        {"params": gradual_model.fc.parameters(), "lr": 1e-3},
    ],
    momentum=0.9,
    weight_decay=1e-4,
)

_, gradual_trainable = count_parameters(gradual_model)
print(f"gradually unfrozen parameters: {gradual_trainable:,}")


## 11. BatchNorm을 고정한다는 것의 의미

BatchNorm에는 두 종류의 상태가 있다.

- 학습 가능한 affine 파라미터: `weight(γ)`, `bias(β)`
- gradient로 학습하지 않는 running statistics: `running_mean`, `running_var`

`requires_grad=False`는 affine 파라미터의 gradient를 막지만 `model.train()` 상태에서는 running statistics가 계속 갱신될 수 있다. 작은 batch에서는 통계가 불안정해질 수 있지만, 목표 domain이 크게 다르면 새 통계에 적응하는 것이 유리할 수도 있다.

모든 BatchNorm을 평가 모드로 고정하고 싶다면 `model.train()`을 호출한 뒤 별도로 다음 함수를 적용할 수 있다. 이는 선택 가능한 정책이며 무조건 적용해야 하는 규칙은 아니다.


In [ ]:
def set_batchnorm_eval(module):
    if isinstance(module, nn.modules.batchnorm._BatchNorm):
        module.eval()


# 사용 예시:
# model.train()
# model.apply(set_batchnorm_eval)
# 이후 training loop를 실행한다.


In [ ]:
def plot_history(history, title):
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history["train_loss"], label="train")
    axes[0].plot(epochs, history["val_loss"], label="validation")
    axes[0].set_title(f"{title} - Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, history["train_accuracy"], label="train")
    axes[1].plot(epochs, history["val_accuracy"], label="validation")
    axes[1].set_title(f"{title} - Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


plot_history(feature_history, "Fixed Feature Extractor")
plot_history(finetune_history, "Full Fine-tuning")


## 12. 학습 곡선 해석

- **Train과 validation loss가 함께 감소:** 정상적인 학습 가능성이 높다.
- **Train loss는 감소하지만 validation loss는 증가:** 과적합 가능성이 있다. Augmentation, weight decay, early stopping, 더 작은 모델을 검토한다.
- **둘 다 거의 감소하지 않음:** 학습률, label, 입력 정규화, 고정 범위, 데이터 품질을 확인한다.
- **Validation accuracy가 크게 흔들림:** Validation sample이 적거나 클래스 불균형, BatchNorm, 높은 학습률의 영향을 확인한다.
- **Fine-tuning 직후 성능이 급락:** Backbone learning rate가 너무 크거나 전처리가 pretrained weight와 맞지 않을 수 있다.

클래스가 불균형하면 accuracy만으로 성능을 판단하지 말고 클래스별 precision, recall, macro F1, confusion matrix를 함께 확인한다.


## 13. 새로운 이미지 추론

추론에서는 `model.eval()`과 `torch.inference_mode()`를 함께 사용한다. 전자는 Dropout과 BatchNorm의 동작을 바꾸고, 후자는 gradient 기록을 중지한다.

예측 클래스만 필요하면 logit에 바로 `argmax`를 적용할 수 있다. 클래스별 확률도 확인하려면 softmax를 적용한다.


In [ ]:
def predict_image(model, image_path, transform, class_names, device):
    model.eval()

    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.inference_mode():
        logits = model(image_tensor)
        probabilities = logits.softmax(dim=1)
        confidence, predicted_index = probabilities.max(dim=1)

    return {
        "class_index": predicted_index.item(),
        "class_name": class_names[predicted_index.item()],
        "confidence": confidence.item(),
        "probabilities": probabilities.squeeze(0).cpu(),
    }


# 사용 예시
# result = predict_image(
#     finetune_model,
#     image_path="sample.jpg",
#     transform=val_transform,
#     class_names=class_names,
#     device=device,
# )
# print(result["class_name"], f"{result['confidence']:.2%}")


## 14. 모델 저장과 재로딩

전체 Python model 객체보다 `state_dict`와 모델 재구성에 필요한 metadata를 저장하는 방식을 권장한다. 클래스 순서도 반드시 저장해야 출력 index를 올바른 이름으로 해석할 수 있다.


In [ ]:
checkpoint_path = Path("resnet18_transfer_learning.pt")

checkpoint = {
    "architecture": "resnet18",
    "num_classes": num_classes,
    "class_names": class_names,
    "model_state_dict": finetune_model.state_dict(),
}

torch.save(checkpoint, checkpoint_path)
print(f"saved to: {checkpoint_path.resolve()}")


In [ ]:
loaded_checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False,
)

loaded_model = resnet18(weights=None)
loaded_model.fc = nn.Linear(
    loaded_model.fc.in_features,
    loaded_checkpoint["num_classes"],
)
loaded_model.load_state_dict(
    loaded_checkpoint["model_state_dict"]
)
loaded_model = loaded_model.to(device)
loaded_model.eval()

loaded_class_names = loaded_checkpoint["class_names"]
print(loaded_class_names)


## 15. 전문가 관점: Domain Shift와 Negative Transfer

원본 데이터 분포를 $P_S(X,Y)$, 목표 데이터 분포를 $P_T(X,Y)$라고 하자. 두 분포가 다르면 domain shift가 존재한다.

$$
P_S(X,Y)\neq P_T(X,Y)
$$

자연 이미지로 학습한 feature가 의료 영상, 적외선, 현미경 영상에 그대로 최적인 것은 아니다. 전이학습이 무작위 초기화보다 오히려 성능을 낮추는 현상을 negative transfer라고 한다.

다음 실험을 통해 판단해야 한다.

- Random initialization baseline
- Classifier-only feature extraction
- 상위 stage 일부 fine-tuning
- 전체 fine-tuning
- 목표 domain과 더 가까운 데이터로 사전 학습한 weight

전이학습의 효과는 모델 이름만으로 결정되지 않고 원본·목표 데이터의 유사성, 데이터 수, label 정의, augmentation과 optimization에 의해 결정된다.


## 16. 자주 발생하는 오류

### 16.1 `pretrained=True` 사용

과거 API 대신 현재 torchvision의 명시적인 weight enum을 사용한다.

```python
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)
```

### 16.2 마지막 출력 수를 변경하지 않음

새 데이터 클래스가 5개인데 ImageNet용 1000개 출력을 유지하면 목표 label 정의와 맞지 않는다.

### 16.3 Softmax 후 CrossEntropyLoss 적용

`CrossEntropyLoss`에는 softmax 이전 logit을 전달한다.

### 16.4 Optimizer 생성 후 파라미터 교체

새 classifier를 만든 다음 optimizer를 생성해야 한다. Optimizer가 이전 layer 파라미터를 계속 참조하지 않도록 한다.

### 16.5 Unfreezing 후 optimizer 갱신 누락

`requires_grad=True`만 변경해도 기존 optimizer에 새 파라미터가 자동으로 추가되지는 않는다. Optimizer를 다시 만들거나 parameter group을 추가한다.

### 16.6 입력 전처리 불일치

RGB/BGR 순서, pixel 범위, resize, normalization을 확인한다. OpenCV의 `cv2.imread`는 기본적으로 BGR이다.

### 16.7 Validation에 무작위 증강 사용

평가 결과가 실행마다 달라질 수 있다. Validation은 결정적인 transform을 사용한다.

### 16.8 가장 마지막 epoch만 저장

과적합 이후의 모델일 수 있다. Validation metric이 가장 좋았던 state를 저장한다.


## 17. 실무 체크리스트

### 데이터

- Train/validation/test 사이에 같은 원본 이미지나 동일 대상이 중복되지 않는가?
- 클래스 index와 폴더 이름이 학습·검증에서 같은가?
- 증강이 label의 의미를 보존하는가?
- 클래스 불균형과 중복 이미지를 확인했는가?

### 모델

- 마지막 출력 수가 `num_classes`와 같은가?
- 실제 trainable parameter를 출력해 확인했는가?
- Backbone과 classifier의 학습률을 분리할 필요가 있는가?
- BatchNorm running statistics 정책을 결정했는가?

### 평가와 배포

- Accuracy뿐 아니라 클래스별 recall과 confusion matrix를 확인했는가?
- Best validation checkpoint를 사용했는가?
- 추론 전처리가 validation 전처리와 일치하는가?
- 목표 GPU·CPU·모바일 장치에서 latency와 memory를 측정했는가?


## 18. 핵심 정리

1. 전이학습은 큰 원본 데이터에서 학습한 표현을 새로운 작업에 재사용한다.
2. 고정 feature extraction은 backbone을 고정하고 새 classifier만 학습한다.
3. Fine-tuning은 backbone 일부 또는 전체를 작은 학습률로 조정한다.
4. 새로운 클래스 수에 맞춰 마지막 출력층을 반드시 교체한다.
5. `requires_grad=False`와 `model.eval()`은 서로 다른 기능이다.
6. 새 classifier와 pretrained backbone에는 서로 다른 learning rate를 사용할 수 있다.
7. Unfreezing 후에는 optimizer가 새 파라미터를 포함하도록 갱신한다.
8. Pretrained weight가 기대하는 입력 전처리를 맞춰야 한다.
9. Validation best state를 저장하고 test set은 최종 평가에만 사용한다.
10. 전이학습도 negative transfer가 가능하므로 scratch baseline과 실제 검증 결과로 판단한다.


## 19. 참고 자료

- [PyTorch Transfer Learning for Computer Vision Tutorial](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- [Torchvision Models and Pre-trained Weights](https://docs.pytorch.org/vision/stable/models.html)
- [Torchvision ResNet-18 Weight Documentation](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html)
- [Residual Connections 상세 설명](./01.Residual_connections.md)

공식 문서의 `weights=` API와 weight별 transform 정보를 확인하면 torchvision 버전에 따른 deprecated 경고와 전처리 불일치를 줄일 수 있다.
